In [ ]:
# importing necessary libraries
import os
import pandas as pd
from IPython.display import Markdown , display
from openai import OpenAI
import json
from enum import Enum
from pydantic import BaseModel,Field
from typing import List,Literal,Optional

In [ ]:
model = "gpt-4.1-mini"

client = OpenAI(base_url="https://openai.vocareum.com/v1",api_key = "voc-915017058160736494244569c3cfb36e0f58.44489188")

In [ ]:
def get_completion (messages=None, system_prompt=None, user_prompt=None, model=model):
  """Function to get a completion from the OpenAI API.
  Args:
  system_prompt: The system prompt
  user_prompt: The user prompt
  model: The model to use (default is gpt-4.1-mini)
  Returns:
  The completion text"""
  messages_to_send = list(messages or [])
  if system_prompt:
    messages_to_send.insert(0, {"role": "system", "content": system_prompt})
  if user_prompt:
    messages_to_send.append({"role": "user", "content": user_prompt})

  # Ensure messages_to_send is not empty before calling the API
  if not messages_to_send:
      raise ValueError("No messages provided to get_completion.")

  response = None # Initialize response to None to ensure it's always defined
  try:
    response = client.chat.completions.create(
      model=model,
      messages=messages_to_send,
      temperature=0.7,
      )
  except Exception as e:
    # Catch potential exceptions during API call.
    # If an error occurs here, response might still be None.
    raise RuntimeError(f"Failed to get completion from OpenAI API: {e}") from e

  # At this point, response should be an object if no exception was raised.
  # If it's still None, it indicates a very unusual scenario.
  if response is None:
      raise RuntimeError("OpenAI API call did not return a response.")

  return response.choices[0].message.content

In [ ]:
# Define sample FNOL texts

sample_fnols = [
    """
    Claim ID: C001
    Customer: John Smith
    Vehicle: 2018 Toyota Camry
    Incident: While driving on the highway, a rock hit my windshield and caused a small chip
    about the size of a quarter. No other damage was observed.
    """,
    """
    Claim ID: C002
    Customer: Sarah Johnson
    Vehicle: 2020 Honda Civic
    Incident: I was parked at the grocery store and returned to find someone had hit my car and
    dented the rear bumper and taillight. The taillight is broken and the bumper has a large dent.
    """,
    """
    Claim ID: C003
    Customer: Michael Rodriguez
    Vehicle: 2022 Ford F-150
    Incident: I was involved in a serious collision at an intersection. The front of my truck is
    severely damaged, including the hood, bumper, radiator, and engine compartment. The airbags
    deployed and the vehicle is not drivable.
    """,
    """
    Claim ID: C004
    Customer: Emma Williams
    Vehicle: 2019 Subaru Outback
    Incident: My car was damaged in a hailstorm. There are multiple dents on the hood, roof, and
    trunk. The side mirrors were also damaged and one window has a small crack.
    """,
    """
    Claim ID: C005
    Customer: David Brown
    Vehicle: 2021 Tesla Model 3
    Incident: Someone keyed my car in the parking lot. There are deep scratches along both doors
    on the driver's side.
    """,
]

<h3>Stage I: Information Extraction</h3>
Creating prompt to extract structured information from free-form FNOL(first notice of loss) text.

In [ ]:
# Define a system prompt for information extraction according to the provided ClaimInformation class


class ClaimInformation(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[
        Literal[
            "windshield",
            "front",
            "rear",
            "side",
            "roof",
            "hood",
            "door",
            "bumper",
            "fender",
            "quarter panel",
            "trunk",
            "glass",
        ]
    ] = Field(..., min_length=1) # Changed min_items to min_length


info_extraction_system_prompt = """
You are an experienced Insurance Claim Manager. Your task is to accurately extract key information from First Notice of Loss (FNOL) reports and present it as a JSON object.

Here is the schema you must follow for the JSON output. Ensure all fields are present and data types match:
```json
{
    "claim_id": "string",
    "name": "string",
    "vehicle": "string",
    "loss_desc": "string",
    "damage_area": [
        "windshield",
        "front",
        "rear",
        "side",
        "roof",
        "hood",
        "door",
        "bumper",
        "fender",
        "quarter panel",
        "trunk",
        "glass"
    ]
}
```

Only respond with the JSON object, nothing else. Do not include any conversational text or explanations.
"""

In [ ]:
# Define a gate check function and claim extraction function


def gate1_validate_claim_info(claim_info_json: str) -> ClaimInformation:
    """
    Gate 1: Validates claim information extracted from FNOL text.
    Returns validated ClaimInformation object or raises validation error.
    """
    try:
        # Parse the JSON string
        claim_info_dict = json.loads(claim_info_json)
        # Validate with Pydantic model
        validated_info = ClaimInformation(**claim_info_dict)
        return validated_info
    except Exception as e:
        raise ValueError(f"Gate 1 validation failed: {str(e)}")


def extract_claim_info(fnol_text):
    """
    Stage 1: Extract structured information from FNOL text
    """
    messages = [
        {"role": "system", "content": info_extraction_system_prompt},
        {"role": "user", "content": fnol_text},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the extracted information
    try:
        validated_info = gate1_validate_claim_info(response) #<-- Run the gate check on the response
        return validated_info
    except ValueError as e:
        print(f"Gate 1 failed: {e}")
        return None

In [ ]:
extracted_claim_info_items = [
    extract_claim_info(fnol_text) for fnol_text in sample_fnols
]
extracted_claim_info_items

<h3>Stage II (Severity Assessment): </h3>Use the extracted information from Stage I as input for a new LLM prompt to assess the damage severity ("Minor," "Moderate," "Major") and estimate repair costs, again with validation.

In [ ]:
# Define a system prompt for severity assessment according to the provided SeverityAssessment class
# TODO: Complete the prompt by replacing the parts marked with **********


class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Your task is to evaluate the severity of vehicle damage and estimate repair costs based on the provided incident information.

Here is the schema you must follow for the JSON output. Ensure all fields are present and data types match:
```json
{
    "severity": "Minor" | "Moderate" | "Major",
    "est_cost": 0.00
}
```

Classify the loss as 'Minor', 'Moderate', or 'Major' and provide an estimated cost.

Only respond with the JSON object, nothing else. Do not include any conversational text or explanations.
"""

In [ ]:
# Define a gate check function and assess_severity function
# TODO: Complete the prompt by replacing the parts marked with **********


class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


def gate2_cost_range_ok(severity_json: str) -> SeverityAssessment:
    """
    Gate 2: Validates that the estimated cost is within reasonable range for the severity.
    Returns validated SeverityAssessment object or raises validation error.
    """
    try:
        # Parse the JSON string
        severity_dict = json.loads(severity_json)
        # Validate with Pydantic model
        validated_severity = SeverityAssessment(**severity_dict)

        # Check cost range based on severity
        if (
            validated_severity.severity == "Minor"
            and not (100 <= validated_severity.est_cost <= 1000) # <-- est_cost outside of heuristic range for Minor will raise ValueError
        ):
            raise ValueError(
                f"Minor damage should cost between $100-$1000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Moderate"
            and not (1000 < validated_severity.est_cost <= 5000) # <-- est_cost outside of heuristic range for Moderate will raise ValueError
        ):
            raise ValueError(
                f"Moderate damage should cost between $1000-$5000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Major"
            and not (5000 < validated_severity.est_cost <= 50000) # <-- est_cost outside of heuristic range for Major will raise ValueError
        ):
            raise ValueError(
                f"Major damage should cost between $5000-$50000, got ${validated_severity.est_cost}"
            )


        return validated_severity
    except Exception as e:
        raise ValueError(f"Gate 2 validation failed: {str(e)}")


def assess_severity(claim_info: ClaimInformation) -> Optional[SeverityAssessment]:
    """
    Stage 2: Assess severity based on damage description
    """

    # Add a check for None before processing claim_info
    if claim_info is None:
        return None

    # Convert Pydantic model to JSON string
    claim_info_json =  claim_info.model_dump_json()

    messages = [
        {"role": "system", "content": severity_assessment_system_prompt},
        {"role": "user", "content": claim_info_json},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the severity assessment
    try:
        validated_severity = gate2_cost_range_ok(response)
        return validated_severity
    except ValueError as e:
        print(f"Gate 2 failed: {e}. Response: {response}")
        return None

In [ ]:
severity_assessment_items = [
    assess_severity(item) for item in extracted_claim_info_items
]

severity_assessment_items

<h3>Stage III (Queue Routing):</h3> Take the outputs from the previous stages to prompt the LLM to route the claim to the correct processing queue ("glass," "fast_track," etc.) based on business rules, with a final validation.

In [ ]:
# Define a system prompt for claim routing according to the provided ClaimRouting class


class ClaimRouting(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss"]
    priority: int = Field(..., ge=1, le=5)


queue_routing_system_prompt = """
You are an auto insurance claim routing specialist. Your task is to determine the appropriate processing queue and priority for each claim based on the provided claim information and severity assessment.

Here is the schema you must follow for the JSON output. Ensure all fields are present and data types match:
```json
{
    "claim_id": "string",
    "queue": "glass" | "fast_track" | "material_damage" | "total_loss",
    "priority": 1 | 2 | 3 | 4 | 5
}
```

Business rules for routing:
- If damage_area includes 'glass', route to 'glass'.
- For 'Minor' severity, route to 'fast_track'.
- For 'Moderate' severity, route to 'material_damage'.
- For 'Major' severity, route to 'total_loss'.
- Priority: Assign 1 for major/urgent cases, 5 for minor/less urgent cases, and 2-4 for intermediate.

Only respond with the JSON object, nothing else. Do not include any conversational text or explanations.
"""

In [ ]:
# Define a gate check function and assess_severity function
# No updates needed in this cell


def gate3_validate_routing(routing_json: str) -> ClaimRouting:
    """
    Gate 3: Validates that the claim is routed to a valid queue.
    Returns validated ClaimRouting object or raises validation error.
    """
    try:
        # Parse the JSON string
        routing_dict = json.loads(routing_json)
        # Validate with Pydantic model
        validated_routing = ClaimRouting(**routing_dict)
        return validated_routing
    except Exception as e:
        raise ValueError(f"Gate 3 validation failed: {str(e)}")


def route_claim(
    claim_info: ClaimInformation, severity_assessment: Optional[SeverityAssessment]
) -> Optional[ClaimRouting]:
    """
    Stage 3: Route claim to appropriate queue
    """
    if severity_assessment is None:
        return None

    # Create input for the routing model
    routing_input = {
        "claim_info": claim_info.model_dump(),
        "severity_assessment": severity_assessment.model_dump(),
    }

    messages = [
        {"role": "system", "content": queue_routing_system_prompt},
        {"role": "user", "content": json.dumps(routing_input)},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the routing decision
    try:
        validated_routing = gate3_validate_routing(response)
        return validated_routing
    except ValueError as e:
        print(f"Gate 3 failed: {e}. Response: {response}")
        return None

In [ ]:
routed_claim_items = [
    route_claim(claim, severity_assessment)
    for claim, severity_assessment in zip(
        extracted_claim_info_items, severity_assessment_items
    )
]

routed_claim_items

In [ ]:
# No updates needed in this cell

import pandas as pd

records = []
for claim, severity_assessment, routed_claim in zip(
    extracted_claim_info_items, severity_assessment_items, routed_claim_items
):
    # Ensure 'claim' is not None before updating
    if claim:
        record = claim.model_dump()
    else:
        record = {}

    if severity_assessment:
        record.update(severity_assessment.model_dump())

    if routed_claim:
        record.update(routed_claim.model_dump())

    records.append(record)


# Show the entire dataframe since it is not too large
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(records)

df